In [4]:
from typing import Sequence, Optional, Union
from pathlib import Path
import json
import logging
from tqdm import tqdm
import re
from collections import OrderedDict

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_openai.chat_models import ChatOpenAI

from ontology.entities import Section, SectionRequirements
from ontology.utilities import escape_braces, get_section_full_text
from ontology.parsing import WordDocumentLoader, ParsingScheme
from ontology.parsing.parsers.utilities import get_numeric_level


logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler())

In [69]:
# Парсинг документа в sections

def get_sections_from_parsed_documents(documents: list[Document]) -> list[Section]:
    """
    Парсинг документа:
    sections – итоговый список параграфов;
    cur_section – текущий параграф.
    """
    sections = []
    cur_section = Section()

    for doc in documents:
        # Определяем уровень заголовка – строка загаловок или нет
        numeric_level = get_numeric_level(doc.page_content)

        if tuple(doc.metadata["headings"]) == cur_section.headings and numeric_level == -1:
            """
            -1 - обычный текст, если заголовок, то 1,2,3...
            Если заголовок совпадает и он уже был, тогда добавляем текст в текущий параграф.
            """
            cur_section.text = "\n".join((cur_section.text, doc.page_content))
            continue

        # Сохраняем параграф
        if cur_section:
            sections.append(cur_section)

        # Новый параграф
        section = Section.from_langchain_document(doc)

        # Новый заголовк -> новый параграф
        if numeric_level != -1:
            cur_section = section
        else:
            sections.append(section)
            cur_section = Section()

    if cur_section:
        sections.append(cur_section)

    return sections


def parse_sections(
    file_path: Union[str, Path],
    remove_service_chapters: bool = True,
    invalid_headings: list[str] | None = None,
    extract_tables: bool = True,
) -> list[Section]:
    """
    Загрузка документов;
    Извлечение таблиц;
    Удаление разделов (если он в списке тех, которые не нужны)
    """
    loader = WordDocumentLoader(
        file_path,
        parsing_scheme=ParsingScheme.paragraphs,
        extract_tables=extract_tables,
        remove_service_chapters=remove_service_chapters,
        log_with_source=True,
    )

    docs = loader.load()
    sections = get_sections_from_parsed_documents(docs)

    if invalid_headings:
        section_for_delete = []
        for i, section in enumerate(sections):
            delete = False
            break_cycle = False

            for inv_head in invalid_headings:
                if not break_cycle:
                    for sec_head in section.headings:
                        if sec_head in inv_head:
                            break_cycle = True
                            delete = True
                            break
                else:
                    break

            if delete:
                section_for_delete.append(i)

        for i in reversed(section_for_delete):
            sections.pop(i)

    return sections


In [70]:
def _normalize_heading(text: str) -> str:
    """
    Функция для нормализации заголовков:
    - перевод в нижний регистр;
    - удаление пустой строки;
    - проверяет повторяющиеся пробелы, табы и переводы строк -> переводит в один пробел;
    - убирает пробелы по краям
    """
    if not text:
        return ""
    text = text.lower().replace("ё", "е")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _heading_matches(heading: str, patterns: Sequence[str]) -> bool:
    """
    Ищем облассть применения и термины и сокращения
    """
    norm = _normalize_heading(heading)
    return any(pattern in norm for pattern in patterns)


def collect_chapter_text_by_heading_patterns(
    sections: Sequence[Section],
    patterns: Sequence[str],
) -> Optional[str]:
    """
    Собирает полный текст всех sections, относящихся к главе,
    если в цепочке headings section встречается нужный заголовок.

    У дочерних секций в section.headings сохраняются родительские заголовки
    """
    collected: "OrderedDict[str, None]" = OrderedDict()

    for section in sections:
        headings = getattr(section, "headings", ()) or ()
        if any(_heading_matches(h, patterns) for h in headings):
            full_text = get_section_full_text(section).strip()
            if full_text:
                collected[full_text] = None

    if not collected:
        return None

    return "\n\n".join(collected.keys())


def build_global_reference_contexts(sections: Sequence[Section]) -> tuple[Optional[str], Optional[str]]:
    """
    Вытаскиваем "контекст" для LLM
    """
    scope_patterns = [
        "область применения",
    ]

    terms_patterns = [
        "термины, определения и сокращения",
        "термины и определения",
        "термины, определения",
        "сокращения",
    ]

    scope_text = collect_chapter_text_by_heading_patterns(sections, scope_patterns)
    terms_text = collect_chapter_text_by_heading_patterns(sections, terms_patterns)

    return scope_text, terms_text

In [71]:
# Модели для extraction

class RequirementRecord(BaseModel):
    """
    Структура документа
    """
    paragraph: Optional[str] = Field(default=None, description="Номер пункта")
    requirement: str = Field(description="Элементарное требование")
    dependance: Optional[str] = Field(
        default=None,
        description="Текст элементарного требования, от которого зависит текущее требование"
    )

class ExtractedRequirements(BaseModel):
    # Список найденных требований
    reqs: Optional[list[Requirement]] = Field(
        description="Элементарные требования, извлеченные из текста. Если отсутствуют в тексте - null"
    )
    # Пункт к которому относится требование
    paragraph: Optional[str] = Field(
        default=None,
        description="Номер пункта, если имеется, иначе null. Если reqs=null, то paragraph тоже null"
    )

class RequirementExtractionInput(BaseModel):
    """
    Входные данные в запросе к LLM
    """
    paragraph: Optional[str] = Field(default=None, description="Номер пункта")
    main_text: str = Field(description="Основной текст фрагмента")
    prev_text: Optional[str] = Field(default=None, description="Предыдущий фрагмент документа")
    next_text: Optional[str] = Field(default=None, description="Следующий фрагмент документа")
    scope_text: Optional[str] = Field(default=None, description="Глава 'Область применения' — только дополнительный контекст")
    terms_text: Optional[str] = Field(default=None, description="Глава 'Термины, определения и сокращения' — только дополнительный контекст")

In [72]:
def extract_requirement_records(
    self,
    sections: Sequence[Section],
) -> list[RequirementRecord]:
    if not sections:
        return []

    inputs = self.build_inputs_from_sections(sections)          # СОдержит ли вход информацию
    responses = self.batch_extract(inputs)                      # Запрос с входными данными

    flat_records: list[RequirementRecord] = []                  # Список всех требований

    # Текст из документа и АТ
    for section, response in zip(sections, responses):
        if not response or not response.reqs:
            continue

        # Берем номер из параграфа
        paragraph = self._extract_paragraph(section) or response.paragraph

        # Для удаления пустых объектов
        for req in response.reqs:
            if not req or not req.text:
                continue

            # Структура требования в файле
            flat_records.append(
                RequirementRecord(
                    paragraph=paragraph,
                    requirement=req.text,
                    dependance=req.dependance,
                )
            )

    return flat_records

In [74]:
class RequirementExtractor:
    TEMPERATURE = 0.1
    MAX_TOKENS = 5000

    def __init__(
        self,
        openai_base_url: str,
        openai_api_token: str,
        model_name: str = "Qwen2.5-72b-64k",
        temperature: float = TEMPERATURE,
        max_tokens: int = MAX_TOKENS,
        scope_text: Optional[str] = None,
        terms_text: Optional[str] = None,
    ) -> None:
        self.llm = ChatOpenAI(
            openai_api_base=openai_base_url,
            openai_api_key=openai_api_token,
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        )

        # Контекст
        self.scope_text = scope_text
        self.terms_text = terms_text

        # Пайплайн: промпт -> LLM -> ответ LLM
        self.chain = (
            ChatPromptTemplate.from_messages([
                ("system", REQ_SYS_PROMPT),
                ("user", REQ_USER_PROMPT),
            ])
            | self.llm.with_structured_output(ExtractedRequirements)
        )

    def extract_one(self, item: RequirementExtractionInput) -> Optional[ExtractedRequirements]:
        try:
            # Формируем словарь для промпта
            return self.chain.invoke({
                "paragraph": item.paragraph or "",
                "main_text": item.main_text,
                "prev_text": item.prev_text or "",
                "next_text": item.next_text or "",
                "scope_text": item.scope_text or "",
                "terms_text": item.terms_text or "",
            })
        except Exception as e:
            logger.warning(f"[Warning] LLM failed on requirements extraction: {e}")
            return None

    # Извлекаем требования из нескольких секций
    def batch_extract(
        self,
        items: Sequence[RequirementExtractionInput],
    ) -> list[Optional[ExtractedRequirements]]:
        if not items:
            return []

        payloads = [
            {
                "paragraph": item.paragraph or "",
                "main_text": item.main_text,
                "prev_text": item.prev_text or "",
                "next_text": item.next_text or "",
                "scope_text": item.scope_text or "",
                "terms_text": item.terms_text or "",
            }
            for item in items
        ]

        results: list[Optional[ExtractedRequirements]] = []

        # Размер батча
        batch_size = 10
        total = len(payloads)

        with tqdm(total=total, desc="Обработка секций (LLM)", unit="section") as pbar:
            # 1 запрос = 10 текстовых запросов
            for i in range(0, total, batch_size):
                batch_payloads = payloads[i:i + batch_size]
                try:
                    batch_results = self.chain.batch(batch_payloads)
                except Exception as e:
                    logger.warning(
                        f"[Warning] Batch extraction failed on batch starting at index {i}: {e}. "
                        f"Falling back to sequential mode for this batch."
                    )
                    # Если не получилось батчом, то каждый запрос обрабатывается отдельно
                    batch_results = [
                        self.extract_one(
                            RequirementExtractionInput(
                                paragraph=payload.get("paragraph") or None,
                                main_text=payload["main_text"],
                                prev_text=payload.get("prev_text") or None,
                                next_text=payload.get("next_text") or None,
                                scope_text=payload.get("scope_text") or None,
                                terms_text=payload.get("terms_text") or None,
                            )
                        )
                        for payload in batch_payloads
                    ]

                results.extend(batch_results)
                pbar.update(len(batch_payloads))

        return results


    def build_input_from_sections(
        self,
        sections: Sequence[Section],
        index: int,
    ) -> RequirementExtractionInput:
        # Текущий параграф
        current_section = sections[index]
        # Предыдущий параграф
        prev_section = sections[index - 1] if index > 0 else None
        # Следующий параграф
        next_section = sections[index + 1] if index < len(sections) - 1 else None

        return RequirementExtractionInput(
            paragraph=get_section_full_text(current_section),
            main_text=get_section_full_text(current_section),
            prev_text=get_section_full_text(prev_section) if prev_section else None,
            next_text=get_section_full_text(next_section) if next_section else None,
            scope_text=self.scope_text,
            terms_text=self.terms_text,
        )

    # Для прогресс бара
    def build_inputs_from_sections(
        self,
        sections: Sequence[Section],
    ) -> list[RequirementExtractionInput]:
        return [
            self.build_input_from_sections(sections, i)
            for i in tqdm(range(len(sections)), desc="Подготовка секций", unit="section")
        ]

    # Сохранение в json
    def save_requirement_records_to_json(
        self,
        records: Sequence[RequirementRecord],
        output_path: str | Path,
    ) -> None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        data = [record.model_dump() for record in records]

        with output_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        logger.info(f"Saved {len(data)} requirements to {output_path}")

    def extract_requirement_records_and_save(
        self,
        sections: Sequence[Section],
        output_path: str | Path,
    ) -> list[RequirementRecord]:
        records = self.extract_requirement_records(sections)
        self.save_requirement_records_to_json(records, output_path)
        return records

    # СОбираем номер пункта
    @staticmethod
    def _extract_paragraph(section: Section) -> Optional[str]:
        for attr_name in ("paragraph", "number", "clause", "id"):
            value = getattr(section, attr_name, None)
            if value:
                return str(value)
        # Вытаскиваем из текста
        text = get_section_full_text(section)
        if not text:
            return None

        match = re.match(r"^\s*(\d+(?:\.\d+)*)\b", text)
        if match:
            return match.group(1)

        return None

In [75]:
def main():
    BASE_URL = "xxx"
    API_KEY = "xxx"
    MODEL_NAME = "xxx"

    input_docx = Path("СП_251.1325800.2016_с_И1_И2_И3_И4_И5.docx")
    output_json = Path("requirements_with_context.json")

    sections = parse_sections(
        file_path=input_docx,
        remove_service_chapters=True,
        invalid_headings=None,
        extract_tables=True,
    )

    scope_text, terms_text = build_global_reference_contexts(sections)
    logger.info(f"Parsed sections: {len(sections)}")
    logger.info(f"Scope context found: {'yes' if scope_text else 'no'}")
    logger.info(f"Terms context found: {'yes' if terms_text else 'no'}")

    extractor = RequirementExtractor(
        openai_base_url=BASE_URL,
        openai_api_token=API_KEY,
        model_name=MODEL_NAME,
        temperature=0.1,
        max_tokens=80000,
        scope_text=scope_text,
        terms_text=terms_text,
    )

    requirements = extractor.extract_requirement_records_and_save(
        sections=sections,
        output_path=output_json,
    )

    logger.info(f"Extracted sections with requirements: {len(requirements)}")

if __name__ == "__main__":
    main()

[Warning] /Users/olgagavrilenko/PycharmProjects/mcp-ontology/ontology/parsing/parsers/utilities.py:224: "HeadingsExtractingFailedWarning: Can not correctly extract headings due to complex document structure" (in /Users/olgagavrilenko/PycharmProjects/mcp-ontology/experiments/resources/СП_251.1325800.2016_с_И1_И2_И3_И4_И5.docx)
Parsed sections: 630
Parsed sections: 630
Parsed sections: 630
Parsed sections: 630
Parsed sections: 630
Parsed sections: 630
Parsed sections: 630
Parsed sections: 630
Scope context found: yes
Scope context found: yes
Scope context found: yes
Scope context found: yes
Scope context found: yes
Scope context found: yes
Scope context found: yes
Scope context found: yes
Terms context found: yes
Terms context found: yes
Terms context found: yes
Terms context found: yes
Terms context found: yes
Terms context found: yes
Terms context found: yes
Terms context found: yes
Обработка секций (LLM): 100%|██████████| 630/630 [1:00:26<00:00,  5.76s/section]
Saved 1697 requirements

#### Experiment 15 paragraphs

In [33]:
from docx import Document

In [34]:
class LLMAtomicRequirement(BaseModel):
    req_text: str = Field(description="Текст элементарного требования")
    local_relevant_context: Optional[str] = Field(
        default=None,
        description=(
            "Релевантный локальный контекст из выбранного параграфа, "
            "не включая само элементарное требование"
        ),
    )
    global_relevant_context: Optional[str] = Field(
        default=None,
        description=(
            "Релевантный фрагмент полного глобального контекста "
            "для данного элементарного требования"
        ),
    )


class LLMExtractedRequirements(BaseModel):
    reqs: Optional[list[LLMAtomicRequirement]] = Field(
        default=None,
        description="Список элементарных требований с релевантными контекстами или null",
    )
    paragraph: Optional[str] = Field(
        default=None,
        description="Полный выбранный параграф или null",
    )


class RequirementDependence(BaseModel):
    global_: Optional[str] = Field(
        default=None,
        alias="global",
        description="Релевантный глобальный контекст требования",
    )
    local: Optional[str] = Field(
        default=None,
        description="Релевантный локальный контекст требования",
    )

    class Config:
        allow_population_by_field_name = True


class Requirement(BaseModel):
    text: str = Field(description="Текст элементарного требования")
    dependence: RequirementDependence = Field(
        description="Релевантный глобальный и локальный контекст требования"
    )


class FinalExtractedRequirements(BaseModel):
    paragraph: str = Field(description="Полный выбранный параграф")
    reqs: Optional[list[Requirement]] = Field(default=None)


class RequirementExtractionInput(BaseModel):
    paragraph_text: str = Field(description="Полный текст выбранного параграфа")


/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_10036/1720825989.py:30: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class RequirementDependence(BaseModel):


In [35]:
REQ_SYS_PROMPT = """
**Role**
Ты — эксперт по нормативным документам в строительной сфере.

**Task**
Твоя задача — разбить входной текст нормативного документа на **элементарные требования**.

**Элементарное требование** — это минимальная атомарная единица нормы, содержащая одно конкретное ограничение, относящееся к **одной сущности** и **одному её свойству**.

Например:
- ✅ «Высота потолка должна быть не менее 2.7 м.» — элементарное требование.
- ❌ «В помещении должны быть свет и вентиляция.» — не элементарное, так как содержит два требования.

Если у нескольких частей предложения общий предмет регулирования и общая нормативная конструкция, но различаются только однородные элементы перечисления, каждую такую часть выделяй как отдельное элементарное требование, сохраняя общий предмет и общую нормативную конструкцию.


Ниже приведён глобальный контекст документа:

{global_context}

Для каждого извлечённого элементарного требования нужно вернуть:
- `req_text` — текст элементарного требования;
- `local_relevant_context` — релевантный локальный контекст;
- `global_relevant_context` — релевантный глобальный контекст.

Что такое `local_relevant_context`:
`local_relevant_context` — это не любой вспомогательный фрагмент рядом с требованием, а только условие применимости требования: фрагмент, который показывает, что требование действует не всегда, а только в определенной ситуации, для определенного объекта, случая, группы пользователей или проектного сценария.
`req_text` отвечает на вопрос: «что именно требуется / допускается / рекомендуется / должно быть сделано?»;
`local_relevant_context` отвечает на вопрос: «где, когда, при каких обстоятельствах и для чего это требование действует?».

**ВАЖНО**
1) Перед тем как вернуть `local_relevant_context`, проверь: будет ли элементарное требование без этого фрагмента ошибочно восприниматься как общее правило? Если да, этот фрагмент является обязательным `local_relevant_context`. Если нет, возвращай `null`.
2) `local_relevant_context` надо смотреть внутри всего параграфа, а не внутри одного предложения, внутри которого лежит элементарное требование.
3) `local_relevant_context` не должен повторяться в `req_text`. `req_text` – само требование, `local_relevant_context` – условие, при котором это требование выполняется.

Что может быть `local_relevant_context`:
- условие применения нормы;
- ситуация проектирования;
- тип здания, блока, помещения, зоны, группы пользователей;
- область применения требования;
- общий вводный фрагмент, который распространяется на несколько следующих требований;
- внешняя рамка, без которой требование ошибочно выглядит как общее.

Что не является `local_relevant_context`:
- само требование;
- нормативное ядро требования;
- повтор `req_text`;
- короткий служебный обрывок, если в тексте есть более полная внешняя рамка;
- фрагмент, не меняющий область действия требования.

Главное правило для `local_relevant_context`:
если без локального контекста требование можно ошибочно понять как общее правило, хотя в исходном тексте оно действует только в специальной ситуации, то `local_relevant_context` обязателен и должен содержать именно эту специальную ситуацию.

Правила выбора `local_relevant_context`:
- `local_relevant_context` должен содержать только внешний контекст требования: условие, область применения, ситуацию, субъект, объект, рамку действия нормы;
- не включай в `local_relevant_context` нормативное ядро требования;
- не включай слова и конструкции типа: «допускается», «следует», «должен», «не допускается», «рекомендуется», «возможно», «предусматривается», «предусматривать», «включать», «исключить», «сократить», если они уже входят в само требование;
- не возвращай в `local_relevant_context` предложение целиком, если в нём содержится само элементарное требование;
- если после удаления нормативного ядра не остаётся отдельного осмысленного фрагмента, но требование уже само содержит все условия применимости, верни `null`;
- если несколько следующих требований подчинены одному общему условию, повторяй это условие в `local_relevant_context` для каждого такого требования;
- если в тексте есть более полная рамка более высокого уровня, не заменяй её коротким обрывком вроде «по заданию на проектирование».

Что такое `global_relevant_context`:
`global_relevant_context` — это фрагмент глобального контекста, который помогает правильно понять требование по смыслу. Обычно это:
- термины;
- определения;
- сокращения;
- ограничения области применения документа.

Правила выбора `global_relevant_context`:
- если в глобальном контексте есть релевантные термины, определения, сокращения или ограничения области применения, которые помогают понять требование, возвращай их;
- если такой релевантный фрагмент есть, не возвращай `null`;
- не возвращай весь глобальный контекст целиком;
- можно объединять несколько коротких релевантных фрагментов в одну строку, если все они прямо относятся к требованию;
- не придумывай новый глобальный контекст, используй только фрагменты, явно присутствующие в глобальном контексте.

Правила для `req_text`:
- `req_text` должен быть максимально близок к исходному тексту;
- сохраняй порядок слов, грамматическую форму, модальность и синтаксис максимально близко к исходному тексту;
- не переставляй части фразы местами;
- не упрощай и не пересказывай норму;
- не начинай `req_text` со строчной буквы, если в исходном тексте это отдельное предложение;
- если смысл требования теряется без общей нормативной рамки, повторяй эту рамку в каждом элементарном требовании.

Правила обработки:
- извлекай требования только из выбранного параграфа;
- не извлекай требования из глобального контекста;
- не объединяй несколько требований в одно;
- при наличии перечислений каждый пункт превращай в отдельное элементарное требование;
- если перечисление описывает состав или перечень элементов в нормативной конструкции, восстанавливай полную нормативную рамку в каждом `req_text`;
- не превращай чисто назывные элементы в самостоятельные требования без восстановленной нормативной рамки;
- удалять можно только явно служебные элементы: редакционные пометы, артефакты разметки, знаки сносок, повторы служебного текста;
- при обработке таблиц используй заголовки строк, заголовки столбцов и значения ячеек для формирования полного смысла требования;
- не включай в требование сноски, редакционные артефакты и повторяющиеся служебные фрагменты.

Если текст не содержит нормативных требований, верни:
- `"reqs": null`
- `"paragraph": null`

**Output format**:
```json
{{
  "paragraph": "string | null",
  "reqs": [
    {{
      "req_text": "string",
      "local_relevant_context": "string | null",
      "global_relevant_context": "string | null"
    }}
  ] | null
}}

**Examples**

Пример 1
Input:
При проектировании БНК в обособленном здании, размещенном на общем участке с основным зданием ОО, допускается предусматривать сокращенный набор общешкольных помещений. По заданию на проектирование возможно исключить административные помещения, сократить число помещений медблока, физкультурно-оздоровительной группы, информационного центра.

Output:
```json
{{
  "paragraph": null,
  "reqs": [
    {{
      "req_text": "При проектировании БНК в обособленном здании, размещенном на общем участке с основным зданием ОО, допускается предусматривать сокращенный набор общешкольных помещений.",
      "local_relevant_context": null,
      "global_relevant_context": "БНК – блок начальных классов; ОО – общеобразовательная организация (школа)."
    }},
    {{
      "req_text": "По заданию на проектирование возможно исключить административные помещения.",
      "local_relevant_context": "При проектировании БНК в обособленном здании, размещенном на общем участке с основным зданием ОО",
      "global_relevant_context": "БНК – блок начальных классов; ОО – общеобразовательная организация (школа)."
    }},
    {{
      "req_text": "По заданию на проектирование возможно сократить число помещений медблока.",
      "local_relevant_context": "При проектировании БНК в обособленном здании, размещенном на общем участке с основным зданием ОО",
      "global_relevant_context": "БНК – блок начальных классов; ОО – общеобразовательная организация (школа)."
    }},
    {{
      "req_text": "По заданию на проектирование возможно сократить число помещений физкультурно-оздоровительной группы.",
      "local_relevant_context": "При проектировании БНК в обособленном здании, размещенном на общем участке с основным зданием ОО",
      "global_relevant_context": "БНК – блок начальных классов; ОО – общеобразовательная организация (школа)."
    }},
    {{
      "req_text": "По заданию на проектирование возможно сократить число помещений информационного центра.",
      "local_relevant_context": "При проектировании БНК в обособленном здании, размещенном на общем участке с основным зданием ОО",
      "global_relevant_context": "БНК – блок начальных классов; ОО – общеобразовательная организация (школа)."
    }}
  ]
}}
```

Пример 2
Input: При размещении территории ОО на земельном участке с выраженным перепадом рельефа места сопряжения горизонтальных участков ландшафта, расположенных на разной высоте, рекомендуется организовывать в виде естественных откосов с травяным покровом.
Output:
```json
{{
  "paragraph": null,
  "reqs": [
    {{
      "req_text": "Места сопряжения горизонтальных участков ландшафта, расположенных на разной высоте, рекомендуется организовывать в виде естественных откосов с травяным покровом.",
      "local_relevant_context": "При размещении территории ОО на земельном участке с выраженным перепадом рельефа",
      "global_relevant_context": null
    }}
    ]
}}
```

Пример 3
Input:
Допускается совмещение функций спальни и игровой при оборудовании спальни детскими раздвижными кроватями (или другими нестационарными спальными местами). К каждой кровати должен быть предусмотрен удобный для ребенка доступ.

Output:
```json
{{
  "paragraph": "Допускается совмещение функций спальни и игровой при оборудовании спальни детскими раздвижными кроватями (или другими нестационарными спальными местами). К каждой кровати должен быть предусмотрен удобный для ребенка доступ.",
  "reqs": [
    {{
      "req_text": "Дополнительно предусматривается игровая комната площадью не менее 2,5 м2 на одного обучающегося.",
      "local_relevant_context": "Для обучающихся, посещающих группу продленного дня",
      "global_relevant_context": null
    }},
    {{
      "req_text": "Дополнительно рекомендуется предусматривать два спальных помещения (для девочек и мальчиков) площадью не менее 4,0 м2 на одного ребенка.",
      "local_relevant_context": "Для группы продленного дня 1‑го класса",
      "global_relevant_context": null
    }}
  ]
}}
```
Верни только JSON.
""".strip()

REQ_USER_PROMPT = """
Извлеки элементарные требования из выбранного параграфа.

Выбранный параграф:
{paragraph_text}

Верни только JSON.
""".strip()

OUTPUT_JSON_SCHEMA = """
{{
  "paragraph": "string | null",
  "reqs": [
    {{
      "req_text": "string",
      "local_relevant_context": "string | null",
      "global_relevant_context": "string | null"
    }}
  ] | null
}}
""".strip()

In [36]:
def _normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.lower().replace("ё", "е")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _clean_optional_text(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    cleaned = value.strip()
    return cleaned or None

def read_docx_paragraphs(file_path: Union[str, Path]) -> list[str]:
    doc = Document(str(file_path))
    paragraphs: list[str] = []

    for p in doc.paragraphs:
        text = p.text.strip()
        if text:
            paragraphs.append(text)

    return paragraphs


def extract_named_chapter_from_docx(
    file_path: Union[str, Path],
    chapter_number: str,
    chapter_title_pattern: str,
    stop_on_next_top_level_heading: bool = True,
) -> Optional[str]:
    paragraphs = read_docx_paragraphs(file_path)
    if not paragraphs:
        return None

    start_idx: Optional[int] = None
    title_pattern_norm = _normalize_text(chapter_title_pattern)

    for i, paragraph in enumerate(paragraphs):
        norm = _normalize_text(paragraph)

        if re.match(rf"^{re.escape(chapter_number)}\b", paragraph.strip()) and title_pattern_norm in norm:
            start_idx = i
            break

        if title_pattern_norm in norm and start_idx is None:
            if chapter_number == "1" and "область применения" in norm:
                start_idx = i
                break
            if chapter_number == "3" and "термины" in norm:
                start_idx = i
                break

    if start_idx is None:
        return None

    collected: list[str] = []

    for i in range(start_idx, len(paragraphs)):
        p = paragraphs[i]

        if i > start_idx and stop_on_next_top_level_heading:
            if re.match(r"^\s*\d+\s+[А-ЯA-ZЁ]", p):
                break

        collected.append(p)

    text = "\n".join(collected).strip()
    return text or None


def build_global_contexts_from_docx(
    file_path: Union[str, Path],
) -> tuple[Optional[str], Optional[str]]:
    scope_text = extract_named_chapter_from_docx(
        file_path=file_path,
        chapter_number="1",
        chapter_title_pattern="область применения",
    )

    terms_text = extract_named_chapter_from_docx(
        file_path=file_path,
        chapter_number="3",
        chapter_title_pattern="термины",
    )

    return scope_text, terms_text


def combine_global_context(
    scope_text: Optional[str],
    terms_text: Optional[str],
) -> str:
    parts: list[str] = []

    if scope_text:
        parts.append(scope_text)

    if terms_text:
        parts.append(terms_text)

    return "\n\n".join(parts).strip()


In [37]:
class RequirementExtractor:
    TEMPERATURE = 0.1
    MAX_TOKENS = 8000

    def __init__(
        self,
        openai_base_url: str,
        openai_api_token: str,
        model_name: str,
        temperature: float = TEMPERATURE,
        max_tokens: int = MAX_TOKENS,
        scope_text: Optional[str] = None,
        terms_text: Optional[str] = None,
    ) -> None:
        self.llm = ChatOpenAI(
            openai_api_base=openai_base_url,
            openai_api_key=openai_api_token,
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        )

        self.scope_text = scope_text
        self.terms_text = terms_text
        self.full_global_context = combine_global_context(scope_text, terms_text)

        self.output_json_schema = OUTPUT_JSON_SCHEMA
        self.chain = (
            ChatPromptTemplate.from_messages([
                ("system", REQ_SYS_PROMPT),
                ("user", REQ_USER_PROMPT),
            ])
            | self.llm.with_structured_output(LLMExtractedRequirements)
        )

    def build_inputs_from_fragments(
        self,
        fragments: Sequence[str],
    ) -> list[RequirementExtractionInput]:
        items: list[RequirementExtractionInput] = []

        for fragment in tqdm(fragments, desc="Подготовка выбранных фрагментов", unit="fragment"):
            items.append(
                RequirementExtractionInput(
                    paragraph_text=fragment,
                )
            )

        return items

    def extract_one(
        self,
        item: RequirementExtractionInput,
    ) -> Optional[LLMExtractedRequirements]:
        try:
            return self.chain.invoke({
                "paragraph_text": item.paragraph_text,
                "global_context": self.full_global_context if self.full_global_context else "Глобальный контекст отсутствует.",
                "output_json_schema": self.output_json_schema,
            })
        except Exception as e:
            logger.warning(f"[Warning] LLM failed on fragment: {e}")
            return None

    def batch_extract(
        self,
        items: Sequence[RequirementExtractionInput],
        batch_size: int = 3,
    ) -> list[Optional[LLMExtractedRequirements]]:
        if not items:
            return []

        payloads = [
            {
                "paragraph_text": item.paragraph_text,
                "global_context": self.full_global_context if self.full_global_context else "Глобальный контекст отсутствует.",
                "output_json_schema": self.output_json_schema,
            }
            for item in items
        ]

        results: list[Optional[LLMExtractedRequirements]] = []
        total = len(payloads)

        with tqdm(total=total, desc="Извлечение требований (LLM)", unit="fragment") as pbar:
            for i in range(0, total, batch_size):
                batch_payloads = payloads[i:i + batch_size]

                try:
                    batch_results = self.chain.batch(batch_payloads)
                except Exception as e:
                    logger.warning(
                        f"[Warning] Batch extraction failed on batch {i}-{i + len(batch_payloads)}: {e}. "
                        f"Falling back to sequential mode."
                    )
                    batch_results = []
                    for payload in batch_payloads:
                        item = RequirementExtractionInput(
                            paragraph_text=payload["paragraph_text"],
                        )
                        batch_results.append(self.extract_one(item))

                results.extend(batch_results)
                pbar.update(len(batch_payloads))

        return results

    def extract_requirement_records_from_fragments(
        self,
        fragments: Sequence[str],
    ) -> list[dict]:
        if not fragments:
            return []

        inputs = self.build_inputs_from_fragments(fragments)
        responses = self.batch_extract(inputs)

        full_records: list[dict] = []

        for fragment, response in zip(fragments, responses):
            if not response or not response.reqs:
                continue

            reqs_with_context: list[dict] = []

            for req in response.reqs:
                if not req or not req.req_text or not req.req_text.strip():
                    continue

                reqs_with_context.append(
                    {
                        "text": req.req_text.strip(),
                        "dependence": {
                            "global": _clean_optional_text(req.global_relevant_context),
                            "local": _clean_optional_text(req.local_relevant_context),
                        },
                    }
                )

            if reqs_with_context:
                full_records.append(
                    {
                        "paragraph": response.paragraph or fragment,
                        "reqs": reqs_with_context,
                    }
                )

        return full_records

    @staticmethod
    def save_to_json(
        data: Sequence[dict],
        output_path: Union[str, Path],
    ) -> None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with output_path.open("w", encoding="utf-8") as f:
            json.dump(list(data), f, ensure_ascii=False, indent=2)

        logger.info(f"Saved {len(data)} records to {output_path}")

    def extract_and_save(
        self,
        fragments: Sequence[str],
        output_path: Union[str, Path],
    ) -> list[dict]:
        data = self.extract_requirement_records_from_fragments(fragments)
        self.save_to_json(data, output_path)
        return data

In [38]:
# def main():
#     BASE_URL = "xxx"
#     API_KEY = "xxx"
#     MODEL_NAME = "xxx"
#
#     input_docx = Path(
#         "/Users/olgagavrilenko/PycharmProjects/mcp-ontology/experiments/resources/СП_251.1325800.2016_с_И1_И2_И3_И4_И5.docx"
#     )
#     output_json = Path("requirements_selected_fragments_with_global_context_v4.json")
#
#     selected_fragments = [
#         "# Общие требования к зданиям и помещениям общеобразовательных организаций\n## Требования к функциональным группам, составу и площадям помещений\n### Учебные помещения\n7.2.1.7 Для оптимизации учебного пространства и возможности его вариативного использования допускается использование трансформируемых перегородок между учебными помещениями и между учебными и рекреационными помещениями.\n(Введен дополнительно, Изм. № 2).\nУчебные помещения в образовательных организациях, реализующих программы начального общего образования (1–4 классы)\nУченики начальных классов, как правило, обучаются в закрепленных за каждым классом учебных помещениях. Состав учебных помещений на один класс: учебное помещение площадью, соответствующей принятой  форме занятий, рекреация зального типа, санузлы.\nДля обучающихся, посещающих группу продленного дня, дополнительно предусматривается игровая комната площадью не менее 2,5 м2 на одного обучающегося. Для группы продленного дня 1-го класса дополнительно рекомендуется предусматривать два спальных помещения (для девочек и мальчиков) площадью не менее 4,0 м2 на одного ребенка.\nДопускается совмещение функций спальни и игровой при оборудовании спальни детскими раздвижными кроватями (или другими нестационарными спальными местами). К каждой кровати должен быть предусмотрен удобный для ребенка доступ. Площадь спальни-игровой определяется из расчета не менее 2,5 м2 на одно место. Для организации отдельных спален для мальчиков и девочек в общем пространстве спальни-игровой рекомендуется использовать трансформируемые перегородки.\nВ состав выделенного БНК также рекомендуется включать учительскую – методический кабинет для учителей начальной школы из расчета не менее 4,5 м2 на одно рабочее место учителя (разрешается размещать за пределами учебной секции) и одну универсальную студию (комнату труда, моделирования и технической игрушки, изобразительного искусства и природы), площадью не менее 90 м2.\n7.2.2.2, 7.2.2.3 (Измененная редакция, Изм. № 4).\nПри проектировании БНК в обособленном здании, размещенном на общем участке с основным зданием ОО, допускается предусматривать сокращенный набор общешкольных помещений. По заданию на проектирование возможно исключить административные помещения, сократить число помещений медблока, физкультурно-оздоровительной группы, информационного центра.\nЕсли выделенный в отдельное здание БНК соединен с основным зданием отапливаемым переходом, допускается совместное использование общешкольных помещений при условии их достаточной мощности.",
#         "# Требования к инженерному оборудованию зданий общеобразовательных организаций\n## Отопление и вентиляция\n<table> <tbody> <tr><td>Основные помещения </td><td>Расчетная температура воздуха, °С</td><td>Кратность воздухообмена в 1 ч, не менее </td></tr> <tr><td>Классные помещения, учебные кабинеты, лаборатории, актовый зал – лекционная аудитория, помещения для вокальных и музыкальных занятий</td><td>18 </td><td>2, но не менее 20 м3/ч наружного воздуха на одно место </td></tr> <tr><td>Учебные мастерские </td><td>18 </td><td>То же </td></tr> <tr><td>Помещения для дополнительных занятий </td><td>18 </td><td>1,5, но не менее 20 м3/ч наружного воздуха на одно место</td></tr> <tr><td>Спальные комнаты </td><td>20 </td><td>То же </td></tr> </tbody> </table>",
#         "# Общие требования к зданиям и помещениям общеобразовательных организаций\n## Требования к функциональным группам, составу и площадям помещений\n### Группа помещений художественного воспитания\nТаблица 7.3",
#         "# Общие требования к зданиям и помещениям общеобразовательных организаций\n## Общие требования к зданиям общеобразовательных организаций\nВ помещениях, оборудованных системами противодымной вентиляции, все окна должны автоматически закрываться при включении оборудования противодымной вентиляции.",
#         "# Область применения\n1.3 Настоящий свод правил не распространяется на спальные корпуса общеобразовательных организаций, в которых созданы условия для проживания обучающихся в интернате.",
#         "# Общие требования к зданиям и помещениям общеобразовательных организаций\n## Общие требования к зданиям общеобразовательных организаций\nПомещения для административно-технического персонала, родительских организаций, кабинетов специалистов, ведущих прием детей в сопровождении родителей, размещают с учетом общих требований СП 118.13330.",
#         "# Общие требования к зданиям и помещениям общеобразовательных организаций\n## Требования к функциональным группам, составу и площадям помещений\n### Рекреации\n7.2.15.5 Допускается размещение входной группы (вестибюля, гардероба, зоны ожидания родителей, места дежурного и др.) в цокольном этаже с учетом требований к размещению помещений ОО в подвальных и цокольных этажах по СП 2.4.3648, СП 1.13130, а также на этажах выше первого (при размещении здания на сложном рельефе), если разница отметок тротуара перед входом и площадки (крыльца) у входной двери в указанную группу помещений не превышает 1,5 м.\n(Введен дополнительно, Изм. № 3).\n(Измененная редакция, № 4).",
#         "# Естественное и искусственное освещение\n## Общие требования\n<table> <tbody> <tr><td>Наименование помещения/нормируемая освещенность на рабочей поверхности, лк </td><td>Удельная установленная мощность ОУ зданий ОО, Вт/м2, не более </td><td>Удельная установленная мощность ОУ зданий ОО, Вт/м2, не более </td><td>Удельная установленная мощность ОУ зданий ОО, Вт/м2, не более </td><td>Удельная установленная мощность ОУ зданий ОО, Вт/м2, не более </td><td>Удельная установленная мощность ОУ зданий ОО, Вт/м2, не более </td></tr> <tr><td>Наименование помещения/нормируемая освещенность на рабочей поверхности, лк </td><td>Индекс помещения i </td><td>Индекс помещения i </td><td>Индекс помещения i </td><td>Индекс помещения i </td><td>Индекс помещения i </td></tr> <tr><td>Наименование помещения/нормируемая освещенность на рабочей поверхности, лк </td><td>0,6 </td><td>0,8 </td><td>1,25 </td><td>2,0 </td><td>&gt; 3,0 </td></tr> <tr><td>Снарядные, инвентарные, хозяйственные кладовые, душевые, гардеробные, электрощитовые*/Е = 100 </td><td>7 </td><td>6 </td><td>5 </td><td>4 </td><td>3 </td></tr> <tr><td>Главные лестничные клетки/Е = 100 </td><td>12 </td><td>11 </td><td>10 </td><td>9 </td><td>8 </td></tr> <tr><td>Вестибюли и гардеробные уличной одежды, рекреации, крытые бассейны/Е = 150 </td><td>15 </td><td>13,75 </td><td>12,5 </td><td>11 </td><td>10 </td></tr> <tr><td>Актовые залы, киноаудитории, спортивные залы*, столовые/Е = 200 </td><td>18 </td><td>17,5 </td><td>15 </td><td>13,5 </td><td>12 </td></tr> <tr><td>Мастерские по обработке металлов и древесины, эстрады актовых залов, кабинеты и комнаты преподавателей, инструментальная, комната мастера, инструктора/Е = 300 </td><td>25 </td><td>23 </td><td>20 </td><td>18 </td><td>16 </td></tr> <tr><td>Классные комнаты, аудитории, учебные кабинеты, лаборатории ОО, кабинеты информатики и вычислительной техники, кабинеты обслуживающих видов труда для девочек, лаборантские при учебных кабинетах/Е = 400</td><td>30 </td><td>28 </td><td>25 </td><td>22 </td><td>20 </td></tr> <tr><td>Классные комнаты, аудитории, учебные кабинеты, лаборатории ОО, кабинеты технического черчения и рисования/Е = 500 </td><td>42 </td><td>39 </td><td>35 </td><td>31 </td><td>28 </td></tr> <tr><td>* Значения удельной мощности, регламентируемые для неосновных ОУ с нормируемой горизонтальной освещенностью менее 100 лк. </td><td>* Значения удельной мощности, регламентируемые для неосновных ОУ с нормируемой горизонтальной освещенностью менее 100 лк.</td><td>* Значения удельной мощности, регламентируемые для неосновных ОУ с нормируемой горизонтальной освещенностью менее 100 лк.</td><td>* Значения удельной мощности, регламентируемые для неосновных ОУ с нормируемой горизонтальной освещенностью менее 100 лк.</td><td>* Значения удельной мощности, регламентируемые для неосновных ОУ с нормируемой горизонтальной освещенностью менее 100 лк.</td><td>* Значения удельной мощности, регламентируемые для неосновных ОУ с нормируемой горизонтальной освещенностью менее 100 лк.</td></tr> </tbody> </table>",
#         "# Требования к размещению и функциональному составу участка\n6.4.9 При размещении территории ОО на земельном участке с выраженным перепадом рельефа места сопряжения горизонтальных участков ландшафта, расположенных на разной высоте, рекомендуется организовывать в виде естественных откосов с травяным покровом. При организации путей движения следует отдавать предпочтение устройству пологих спусков. Использование механических подъемных устройств для МГН на территории ОО не рекомендуется.\n6.4.8, 6.4.9 (Введены дополнительно, Изм. № 2).\nНа территории ОО выделяют следующие зоны: физкультурно-спортивная, отдыха и хозяйственная. Допускается выделение учебно-опытной зоны.\n(Измененная редакция, Изм. № 2).\nПри проектировании физкультурно-спортивной зоны следует руководствоваться пунктами 2.2.2, 3.4.1 СП 2.4.3648–20, таблицами 5.56 и 5.60 СанПиН 1.2.3685–21, СП 42.13330, СП 332.1325800, [14].\nКомплекс площадок физкультурно-спортивной зоны и их оборудование должны соответствовать образовательным программам, реализуемым ОО.\nПлощади открытых плоскостных сооружений физкультурно-спортивной зоны приведены в таблице 6.1.\n(Измененная редакция, Изм. № 2, № 4).\nТаблица 6.1\n<table> <tbody> <tr><td>Наименование открытого плоскостного физкультурно-спортивного сооружения и его размеры </td><td>Площадь, м2, при числе классов, чел., для школы</td><td>Площадь, м2, при числе классов, чел., для школы</td><td>Площадь, м2, при числе классов, чел., для школы</td><td>Площадь, м2, при числе классов, чел., для школы</td><td>Площадь, м2, при числе классов, чел., для школы</td><td>Площадь, м2, при числе классов, чел., для школы</td><td>Площадь, м2, при числе классов, чел., для школы</td></tr> <tr><td>Наименование открытого плоскостного физкультурно-спортивного сооружения и его размеры </td><td>Начальная и основная школа </td><td>Начальная и основная школа </td><td>Начальная и основная школа </td><td>Полнокомплектная школа </td><td>Полнокомплектная школа </td><td>Полнокомплектная школа </td><td>Полнокомплектная школа </td></tr> <tr><td>Наименование открытого плоскостного физкультурно-спортивного сооружения и его размеры </td><td>один класс в парал-лели </td><td>два класса в парал-лели </td><td>три класса в парал-лели </td><td>один класс в парал-лели </td><td>два класса в парал-лели </td><td>три класса в парал-лели </td><td>Более трех классов в парал-лели </td></tr> <tr><td>Наименование открытого плоскостного физкультурно-спортивного сооружения и его размеры </td><td>9 (225 чел.) </td><td>18 (450 чел.) </td><td>21 (525 чел.) </td><td>11 275 чел.) </td><td>22 (550 чел.) </td><td>33 (825 чел.) </td><td>Более 33 классов (более 1100 чел.) </td></tr> <tr><td>Спортивный стадион * </td><td>4950 </td><td>4950 </td><td>4950 </td><td>4950 </td><td>5260 </td><td>5260 </td><td>5260 </td></tr> <tr><td>Футбольное поле (60×40 м)* </td><td>2604* </td><td>2604* </td><td>2604* </td><td>2604* </td><td>2604* </td><td>2604* </td><td>2604* </td></tr> <tr><td>Круговая беговая дорожка на четыре полосы, длиной не менее 200 м, с прямым участком не менее 118 м </td><td>825* </td><td>825* </td><td>825* </td><td>825* </td><td>825* </td><td>825* </td><td>825* </td></tr> <tr><td>Универсальная площадка для общефизической подготовки и физкультурно-оздоровительных занятий (60×30 м)</td><td>1800 </td><td>1800 </td><td>1800 </td><td>1800 </td><td>1800 </td><td>1800 </td><td>1800 </td></tr> <tr><td>Волейбольная площадка (18×9 м) </td><td>360 </td><td>360 </td><td>360 </td><td>360 </td><td>360 </td><td>360 </td><td>360 </td></tr> <tr><td>Баскетбольная площадка (20×10 м) </td><td>264 </td><td>264 </td><td>264 </td><td>264 </td><td>264 </td><td>264 </td><td>264 </td></tr> <tr><td>Площадка для прыжков в длину*** </td><td>50 </td><td>50 </td><td>50 </td><td>50 </td><td>50 </td><td>50 </td><td>50 </td></tr> <tr><td>Теннисный корт (36,0×18,0 м) </td><td>– </td><td>– </td><td>– </td><td>– </td><td>648** </td><td>648** </td><td>648** </td></tr> <tr><td>Площадка для подвижных игр и общеразвивающих упражнений </td><td>710 </td><td>710 </td><td>710 </td><td>710 </td><td>710 </td><td>710 </td><td>710 </td></tr> <tr><td>Иные площадки </td><td>** </td><td>** </td><td>** </td><td>** </td><td>** </td><td>** </td><td>** </td></tr> <tr><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td><td>* При невозможности размещения на территории либо наличии рядом расположенного стадиона, соответствующего требованиям СП 2.4.3648, СанПиН 1.2.3685, допускается замена спортивного стадиона (спортивного ядра, устраиваемого по [14]) отдельными площадками: круговой беговой дорожкой с прямым участком, площадками для спортивных игр. ** По заданию на проектирование. *** При невозможности размещения круговой беговой дорожки, допускается устройство беговой дорожкой с замкнутым контуром, с радиусами поворотов не менее 13 м и прямым участком длиной не менее 118 м. Устройство беговых дорожек вокруг зданий не допускается. Примечания 1 В таблице приведены минимальные параметры спортивных площадок, допустимые для проведения занятий с обучающимися. 2 Площади игровых полей приведены с учетом необходимого свободного пространства по краю. 3 В условиях сложившейся застройки и дефицита участка территории допускается размещение круговой беговой дорожки как самостоятельного плоскостного сооружения (с пересечением автомобильных проездов к зданию, частичным использованием в качестве пожарного проезда с соблюдением требований к пожарным проездам). </td></tr> </tbody> </table>\nТаблица 6.1 (Измененная редакция, Изм. № 2, № 4).\nЗона отдыха\nДля отдыха на участке рекомендуется предусматривать: - площадки для подвижных игр обучающихся начальной школы (2–4 классы) – из расчета не менее 100 м2 на каждый класс, для обучающихся 1-х классов – не менее 180 м2 (7,2 м2 на одного ученика); для основной школы (5–9 классы) – не менее 25 м2 на каждый класс; - площадки для тихого отдыха обучающихся основной школы принимаются для 75 % обучающихся, оборудуются теневыми навесами и малыми игровыми формами.\nДопускается дополнительно предусматривать площадки для подвижных игр и тихого отдыха групп продленного дня.\nДля обучающихся средней школы зоной отдыха служат площадки физкультурно-спортивной зоны.\n(Измененная редакция, Изм. № 4).\nХозяйственная зона\nХозяйственную зону размещают со стороны входа в помещения загрузочного цеха пищеблока и вблизи учебно-опытной зоны. Состав и площади хозяйственных построек определяют заданием на проектирование.\nХозяйственная зона предназначается для размещения хозяйственных построек, мусоросборников, некапитальных объектов для хранения оборудования и инвентаря (разрешается размещать в подвальном или цокольном этаже здания с отдельным выходом наружу). Навес для инвентаря допускается пристраивать к хозяйственной постройке. Над входами в пищеблок и над загрузочной платформой следует предусматривать навес. Высота навеса должна соответствовать используемому транспорту, габариты навеса должны перекрывать габариты платформы и кузова используемого транспорта не менее чем на 1 м с каждой стороны. С учетом местных условий в хозяйственной зоне допускается размещение овощехранилища.\nЛокальные очистные сооружения и другие сооружения систем жизнеобеспечения здания допускается размещать на участке ОО, если от них не требуется устанавливать санитарно-защитные зоны согласно требованиям СанПиН 2.2.1/2.1.1.1200. Их размещают в хозяйственной зоне с соблюдением необходимых разрывов до зданий и сооружений по требованиям СП 4.13130 и ограждают забором в соответствии с СП 31.13330, СП 32.13330, СП 89.13330, [9].\nХозяйственную зону следует отделять от остальных зон защитной полосой зеленых насаждений. К месту загрузки-выгрузки пищеблока должен быть обеспечен безопасный для обучающихся подъезд транспорта и предусмотрена площадка для его разворота. Въезд в хозяйственную зону рекомендуется предусматривать самостоятельным с улицы или внутриквартального проезда, изолированно от входа обучающихся на территорию ОО.\nВ условиях сложившейся (плотной) городской застройки допускается отсутствие самостоятельного подъезда в хозяйственную зону с улицы при условии организации подъезда автотранспорта к хозяйственной площадке в период отсутствия обучающихся в ОО.\n(Измененная редакция, Изм. № 4).",
#         "# Энергетическая эффективность зданий общеобразовательных организаций\nВ целях установления соответствия теплозащитных и энергетических характеристик здания ОО нормируемым показателям и/или требованиям энергетической эффективности объектов капитального строительства, определяемых федеральным законодательством, в ходе проектирования здания ОО следует разрабатывать «Энергетический паспорт проекта здания», форма для заполнения которого приведена в приложении Д СП 50.13330.2012.",
#         "# 4 Общие положения\nВместимость зданий (расчетное число обучающихся) и организационно-педагогическую структуру ОО устанавливают заданием на проектирование исходя из градостроительных и демографических условий с учетом приложения В.",
#         "# Особенности проектирования комбинированного блока начальных классов\nПримечание – Наполняемость универсальной ячейки (групповая ячейка ДОО – класс) принята 25 человек.",
#         "# Требования к размещению и функциональному составу участка\nОО размещают на собственном земельном участке в отдельно стоящем здании или комплексе зданий, реализующих образовательные программы: отдельно стоящих зданий ОО разных уровней, отдельно стоящих функциональных корпусах ОО, ДОО, ОДО, а также отдельно стоящих корпусов (блоков помещений) и зданий и сооружений жизнеобеспечения. Размещение на территории ОО зданий и сооружений, функционально с ней не связанных, не допускается.",
#         "# 4 Общие положения\nНаполняемость классов и групп продленного дня устанавливают заданием на проектирование. Площади учебных помещений принимают исходя из площади помещения на одного обучающегося по СП 2.4.3648 и расчетного количества мест в учебном помещении.",
#         " # Требования к размещению и функциональному составу участка\nВ соответствии с требованиями СП 2.4.3648 территория должна быть ограждена, отсутствие ограждения допускается только со стороны стен здания, непосредственно прилегающих к проезжей части улицы или пешеходному тротуару. При этом должен быть обеспечен проезд пожарных автомобилей со всех сторон здания в соответствии с требованиями 7.3.5.",
#     ]
#
#     logger.info("Извлечение глобального контекста из docx")
#     scope_text, terms_text = build_global_contexts_from_docx(input_docx)
#
#     logger.info(f"Scope context found: {'yes' if scope_text else 'no'}")
#     logger.info(f"Terms context found: {'yes' if terms_text else 'no'}")
#
#     if not scope_text:
#         logger.warning('Не удалось извлечь главу "1 Область применения"')
#     if not terms_text:
#         logger.warning('Не удалось извлечь главу "3 Термины, определения и сокращения"')
#
#     extractor = RequirementExtractor(
#         openai_base_url=BASE_URL,
#         openai_api_token=API_KEY,
#         model_name=MODEL_NAME,
#         temperature=0.1,
#         max_tokens=80000,
#         scope_text=scope_text,
#         terms_text=terms_text,
#     )
#
#     logger.info("Запуск извлечения требований по выбранным фрагментам")
#     data = extractor.extract_and_save(
#         fragments=selected_fragments,
#         output_path=output_json,
#     )
#
#     logger.info(f"Extracted records: {len(data)}")
#
#
# if __name__ == "__main__":
#     main()

In [46]:
SELECTED_SECTION_IDS = [
    "4.1", "4.3", "4.4", "5.3", "5.4",
    "6.4", "6.4.4", "6.4.5", "7.1.3", "7.1.7",
    "4.6", "6.1", "6.2", "6.3", "6.4.2",
    "6.4.3", "6.4.7", "7.1.2", "7.1.9", "7.2.1.3", "7.2.2.4",
]


def load_selected_sections_from_json(
    json_path: Union[str, Path],
    selected_section_ids: Sequence[str],
) -> list[str]:
    json_path = Path(json_path)

    with json_path.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    records = payload["records"]
    selected = set(selected_section_ids)

    fragments: list[str] = []
    seen: set[str] = set()

    for record in records:
        clause_id = record.get("clause_id")

        if clause_id not in selected:
            continue

        # Для длинных section_chunk, например 6.3, берем полный текст секции
        text = record.get("parent_section_text") or record.get("paragraph_text")

        if not text:
            continue

        text = re.sub(r"\s+", " ", text).strip()

        if not text or clause_id in seen:
            continue

        seen.add(clause_id)

        heading_path = record.get("heading_path") or []
        headings = "\n".join(f"# {h}" for h in heading_path)

        if headings:
            fragments.append(f"{headings}\n{clause_id} {text}")
        else:
            fragments.append(f"{clause_id} {text}")

    return fragments

In [47]:
def main():
    BASE_URL = "xxx"
    API_KEY = "xxx"
    MODEL_NAME = "xxx"


    input_docx = Path(
        "itmo-practice-spring-2026/data/СП_251.1325800.2016_с_И1_И2_И3_И4_И5.docx"
    )

    selected_sections_json = Path(
        "itmo-practice-spring-2026/parsed_fragments_only_v27_СП_251.json"
    )

    output_json = Path("requirements_selected_with_global_context_qwen.json")

    selected_fragments = load_selected_sections_from_json(
        json_path=selected_sections_json,
        selected_section_ids=SELECTED_SECTION_IDS,
    )

    logger.info(f"Loaded selected sections: {len(selected_fragments)}")

    logger.info("Извлечение глобального контекста из docx")
    scope_text, terms_text = build_global_contexts_from_docx(input_docx)

    if not scope_text:
        logger.warning('Не удалось извлечь главу "1 Область применения"')
    if not terms_text:
        logger.warning('Не удалось извлечь главу "3 Термины, определения и сокращения"')

    extractor = RequirementExtractor(
        openai_base_url=BASE_URL,
        openai_api_token=API_KEY,
        model_name=MODEL_NAME,
        temperature=0.1,
        max_tokens=8000,
        scope_text=scope_text,
        terms_text=terms_text,
    )

    logger.info("Запуск извлечения требований по выбранным фрагментам")
    data = extractor.extract_and_save(
        fragments=selected_fragments,
        output_path=output_json,
    )

    logger.info(f"Extracted records: {len(data)}")


if __name__ == "__main__":
    main()

Loaded selected sections: 21
Извлечение глобального контекста из docx
Запуск извлечения требований по выбранным фрагментам
Извлечение требований (LLM): 100%|██████████| 21/21 [07:03<00:00, 20.18s/fragment]
Saved 19 records to requirements_selected_with_global_context_qwen.json
Extracted records: 19
